Tuesday, hands on: the report you would sign, solved

> "If there is a gap, I want to know which orders and which channel."

Released at close of session. Every step takes the row count before it takes any total,
which is the whole discipline of the day.

In [1]:
import pathlib
import sys

root = next(p for p in pathlib.Path.cwd().resolve().parents if (p / "scripts" / "c2kit.py").exists())
sys.path.insert(0, str(root / "scripts"))
import c2kit as kit

conn = kit.connect()
kit.flow(["attach", "count", "explain", "collapse", "report"], lit=[0], title="Where you are")

## 1. Rows before

How many rows are in `orders`, and how many in `payments`? Write both down on paper before you
run anything else.

In [2]:
before = kit.sql("""SELECT 'orders' AS t, count(*) AS n FROM orders UNION ALL SELECT 'payments', count(*) FROM payments""", conn=conn)
kit.check("both table sizes came back", len(before) == 2, str(before))

## 2. Rows after

LEFT JOIN orders to payments and count. Do not sum anything yet.

In [3]:
after = kit.sql("""SELECT count(*) AS join_rows FROM orders o LEFT JOIN payments p ON p.order_id = o.order_id""", conn=conn)[0]
n = list(after.values())[0]
kit.check("the join returned 1,450 rows", n == 1450, f"{n} rows")
kit.vflow(["1,000 orders", "LEFT JOIN payments", f"{n} rows", "explain the difference"],
          lit=[3], title="The count check")

## 3. Explain the difference

How many orders carry more than one payment row, and how many carry none? Those two numbers
plus 1,000 have to account for 1,450 exactly.

In [4]:
multi = kit.sql("""SELECT count(*) AS n FROM (SELECT order_id FROM payments GROUP BY order_id HAVING count(*) > 1) t""", conn=conn)[0]
none = kit.sql("""SELECT count(*) AS n FROM orders o LEFT JOIN payments p ON p.order_id = o.order_id WHERE p.payment_id IS NULL""", conn=conn)[0]
m, z = list(multi.values())[0], list(none.values())[0]
print(f"orders with more than one payment: {m}")
print(f"orders with no payment at all    : {z}")
kit.check("the arithmetic accounts for every row", 1000 + m == 1450, f"1000 + {m}")

orders with more than one payment: 450
orders with no payment at all    : 30


## 4. Which orders were never paid

Anand asked for these by name. Keep every order, then keep only the ones that failed to match.

In [5]:
unpaid = kit.sql("""SELECT o.order_id, o.channel, o.amount FROM orders o
       LEFT JOIN payments p ON p.order_id = o.order_id
       WHERE p.payment_id IS NULL ORDER BY o.amount DESC""", conn=conn)
kit.check("thirty orders were never paid", len(unpaid) == 30, f"{len(unpaid)}")
kit.ladder(["every order", "LEFT JOIN payments", "keep the NULL side", "the unpaid list"],
           lit=[2], title="An anti-join finds absence")

## 5. Retries against instalments

Two payment rows with the same amount is a repeated charge. Two rows with different amounts is a
split invoice, which is correct. A rule that deleted every duplicate would delete four hundred
legitimate instalments.

In [6]:
kinds = kit.sql("""WITH doubled AS (
           SELECT order_id,
                  CASE WHEN count(DISTINCT amount) = 1 THEN 'retry, the same amount twice'
                       ELSE 'instalment plan, different amounts' END AS kind
           FROM payments GROUP BY order_id HAVING count(*) > 1)
       SELECT kind, count(*) AS orders FROM doubled GROUP BY kind ORDER BY orders DESC""", conn=conn)
kit.table(list(kinds[0]) if kinds else ["kind"], [list(r.values()) for r in kinds],
          caption="The 450, separated")
kit.check("four hundred are instalment plans",
          any(400 in [v for v in r.values() if isinstance(v, int)] for r in kinds), str(kinds))

kind,orders
"instalment plan, different amounts",400
"retry, the same amount twice",50


## 6. The number you would sign

Collapse the many side to one row per order, then join. Rows out must be 1,000.

In [7]:
real = kit.sql("""WITH paid AS (SELECT order_id, sum(amount) AS collected FROM payments GROUP BY order_id)
       SELECT count(*) AS rows_out, sum(o.amount) AS booked,
              coalesce(sum(p.collected), 0) AS collected
       FROM orders o LEFT JOIN paid p ON p.order_id = o.order_id""", conn=conn)[0]
kit.check("one row in, one row out", real["rows_out"] == 1000, str(real["rows_out"]))
kit.decision_ladder(["sum over the raw join", "sum and hope", "aggregate first, then join",
                     "aggregate, join, and state the count"], cut_at=2,
                    title="What reaches Finance")

## 7. The gap, by channel

Q2 only, one row per channel, carrying booked, collected and the difference. Watch what happens
to a channel containing an unpaid order if you leave `coalesce` out.

In [8]:
gap = kit.sql("""WITH paid AS (SELECT order_id, sum(amount) AS collected FROM payments GROUP BY order_id)
       SELECT o.channel, sum(o.amount) AS booked,
              coalesce(sum(p.collected), 0) AS collected,
              sum(o.amount) - coalesce(sum(p.collected), 0) AS gap
       FROM orders o LEFT JOIN paid p ON p.order_id = o.order_id
       WHERE o.quarter = 'Q2' GROUP BY o.channel ORDER BY gap DESC""", conn=conn)
kit.check("one row per channel", len(gap) == 3, f"{len(gap)} rows")
kit.table(list(gap[0]) if gap else ["channel"], [list(r.values()) for r in gap],
          caption="Q2 booked against collected")
kit.check_summary()

channel,booked,collected,gap
store,32148730.00,31196760.00,951970.00
web,23661000.00,22879290.00,781710.00
app,42590270.00,42589770.00,500.00
